# Data Preprocessing And CV Fluency Drills

This notebook is for IOAI-style preprocessing speed. It is not a lecture and it is not a solution bank.

Use it like contest gym work:

- Run the setup cell.
- For each drill, replace only `answer = None`.
- Predict the shape/dtype before running the check.
- If a check fails, inspect shape, dtype, columns, index alignment, and disk reloads first.

Most drills should take 1-5 minutes. Blank answers are safe, so the whole notebook can run before you complete it.


## Setup

This cell creates tiny local CSV, JSON, JSONL, NumPy, text, image, mask, and annotation files under `_drill_data/`.
Re-run it whenever you want a clean reset.


In [1]:
from __future__ import annotations

import csv
import json
import math
import random
import re
import shutil
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageEnhance, ImageFilter, ImageOps

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
except Exception as exc:
    torch = None
    nn = None
    F = None
    print("PyTorch import failed. Tensor/model sections need torch:", repr(exc))

try:
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.feature_extraction.text import CountVectorizer
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import accuracy_score
    from sklearn.model_selection import train_test_split
except Exception as exc:
    RandomForestClassifier = None
    CountVectorizer = None
    LogisticRegression = None
    accuracy_score = None
    train_test_split = None
    print("scikit-learn import failed. Model sections need sklearn:", repr(exc))


def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    if torch is not None:
        torch.manual_seed(seed)


seed_everything(42)
np.set_printoptions(precision=3, suppress=True)

DATA_DIR = Path("_drill_data")


def reset_drill_data(data_dir: Path = DATA_DIR) -> None:
    if data_dir.exists():
        shutil.rmtree(data_dir)
    data_dir.mkdir(parents=True, exist_ok=True)


def make_drill_data(data_dir: Path = DATA_DIR) -> dict[str, Path]:
    reset_drill_data(data_dir)
    (data_dir / "images").mkdir()
    (data_dir / "masks").mkdir()

    train_rows = [
        {"sample_id": "s001", "team": "red", "age": 15, "height_cm": 172, "score": 88.0, "hours": 7.5, "device": "laptop", "label": 1},
        {"sample_id": "s002", "team": "blue", "age": 16, "height_cm": 168, "score": 71.0, "hours": 4.0, "device": "tablet", "label": 0},
        {"sample_id": "s003", "team": "red", "age": 15, "height_cm": 181, "score": 93.0, "hours": 8.0, "device": "laptop", "label": 1},
        {"sample_id": "s004", "team": "green", "age": 17, "height_cm": 159, "score": 55.0, "hours": 2.0, "device": "phone", "label": 0},
        {"sample_id": "s005", "team": "blue", "age": 16, "height_cm": 175, "score": np.nan, "hours": 5.5, "device": "laptop", "label": 1},
        {"sample_id": "s006", "team": "green", "age": 18, "height_cm": 190, "score": 42.0, "hours": 1.0, "device": "tablet", "label": 0},
        {"sample_id": "s007", "team": "red", "age": 14, "height_cm": 165, "score": 77.0, "hours": 6.0, "device": "phone", "label": 1},
        {"sample_id": "s008", "team": "blue", "age": 19, "height_cm": 210, "score": 64.0, "hours": 3.5, "device": "laptop", "label": 0},
        {"sample_id": "s009", "team": "green", "age": 17, "height_cm": 170, "score": 81.0, "hours": 7.0, "device": "tablet", "label": 1},
        {"sample_id": "s010", "team": "red", "age": 18, "height_cm": 178, "score": 69.0, "hours": 4.5, "device": "laptop", "label": 0},
        {"sample_id": "s011", "team": "blue", "age": 15, "height_cm": 162, "score": 90.0, "hours": 8.5, "device": "phone", "label": 1},
        {"sample_id": "s012", "team": "green", "age": 16, "height_cm": 174, "score": 59.0, "hours": 2.5, "device": "tablet", "label": 0},
    ]
    test_rows = [
        {"sample_id": "t001", "team": "red", "age": 15, "height_cm": 171, "score": 83.0, "hours": 6.5, "device": "laptop"},
        {"sample_id": "t002", "team": "blue", "age": 17, "height_cm": 169, "score": 61.0, "hours": 3.0, "device": "tablet"},
        {"sample_id": "t003", "team": "green", "age": 16, "height_cm": 177, "score": np.nan, "hours": 5.0, "device": "phone"},
        {"sample_id": "t004", "team": "red", "age": 19, "height_cm": 184, "score": 74.0, "hours": 4.0, "device": "laptop"},
    ]
    train_df = pd.DataFrame(train_rows)
    test_df = pd.DataFrame(test_rows)
    train_df.to_csv(data_dir / "train.csv", index=False)
    test_df.to_csv(data_dir / "test.csv", index=False)
    pd.DataFrame({"sample_id": test_df["sample_id"], "label": 0}).to_csv(data_dir / "sample_submission.csv", index=False)

    records = [
        {"id": "r001", "label": "cat", "meta": {"score": 0.82, "size": [32, 32]}, "tags": ["small", "fur"]},
        {"id": "r002", "label": "dog", "meta": {"score": 0.55, "size": [48, 32]}, "tags": ["fur"]},
        {"id": "r003", "label": "cat", "meta": {"score": 0.91, "size": [40, 40]}, "tags": ["small", "indoor"]},
        {"id": "r004", "label": "bird", "meta": {"score": 0.37, "size": [24, 24]}, "tags": []},
        {"id": "r005", "label": "dog", "meta": {"score": 0.73, "size": [64, 48]}, "tags": ["outdoor"]},
    ]
    (data_dir / "records.json").write_text(json.dumps(records, indent=2), encoding="utf-8")
    with (data_dir / "events.jsonl").open("w", encoding="utf-8") as f:
        for row in [
            {"user": "u1", "event": "view", "value": 1},
            {"user": "u1", "event": "click", "value": 3},
            {"user": "u2", "event": "view", "value": 1},
            {"user": "u3", "event": "view", "value": 1},
            {"user": "u2", "event": "click", "value": 2},
        ]:
            f.write(json.dumps(row) + "\n")

    arr = np.arange(24).reshape(4, 6)
    vecs = np.array([[3., 4.], [1., 0.], [0., 0.], [5., 12.]], dtype=np.float32)
    np.save(data_dir / "matrix.npy", arr)
    np.savez(data_dir / "arrays.npz", arr=arr, vecs=vecs)

    texts = pd.DataFrame(
        {
            "text_id": ["tx1", "tx2", "tx3", "tx4", "tx5", "tx6"],
            "text": [
                "Cats chase bright toys.",
                "Dogs chase balls outside.",
                "Bright birds sing loudly.",
                "Cats nap inside quietly.",
                "Dogs guard the yard.",
                "Birds fly outside.",
            ],
            "label": [1, 0, 1, 1, 0, 0],
        }
    )
    texts.to_csv(data_dir / "texts.csv", index=False)

    image_rows = []
    specs = [
        ("img_000.png", "square", 0, (8, 8, 31, 31), (220, 50, 50)),
        ("img_001.png", "circle", 1, (16, 10, 43, 37), (50, 90, 220)),
        ("img_002.png", "square", 0, (4, 16, 27, 43), (220, 50, 50)),
        ("img_003.png", "circle", 1, (20, 18, 47, 45), (50, 90, 220)),
        ("img_004.png", "square", 0, (12, 4, 39, 31), (220, 50, 50)),
        ("img_005.png", "circle", 1, (10, 20, 37, 47), (50, 90, 220)),
        ("img_006.png", "square", 0, (18, 8, 45, 35), (220, 50, 50)),
        ("img_007.png", "circle", 1, (6, 6, 35, 35), (50, 90, 220)),
    ]
    for name, shape, label, bbox, color in specs:
        img = Image.new("RGB", (56, 56), (20, 20, 20))
        mask = Image.new("L", (56, 56), 0)
        d_img = ImageDraw.Draw(img)
        d_mask = ImageDraw.Draw(mask)
        if shape == "square":
            d_img.rectangle(bbox, fill=color)
            d_mask.rectangle(bbox, fill=255)
        else:
            d_img.ellipse(bbox, fill=color)
            d_mask.ellipse(bbox, fill=255)
        img.save(data_dir / "images" / name)
        mask_name = name.replace("img_", "mask_")
        mask.save(data_dir / "masks" / mask_name)
        x1, y1, x2, y2 = bbox
        image_rows.append(
            {
                "image_id": name,
                "mask_id": mask_name,
                "label": label,
                "shape": shape,
                "x1": x1,
                "y1": y1,
                "x2": x2,
                "y2": y2,
            }
        )
    pd.DataFrame(image_rows).to_csv(data_dir / "image_manifest.csv", index=False)

    return {
        "train_csv": data_dir / "train.csv",
        "test_csv": data_dir / "test.csv",
        "sample_submission_csv": data_dir / "sample_submission.csv",
        "records_json": data_dir / "records.json",
        "events_jsonl": data_dir / "events.jsonl",
        "matrix_npy": data_dir / "matrix.npy",
        "arrays_npz": data_dir / "arrays.npz",
        "texts_csv": data_dir / "texts.csv",
        "image_manifest_csv": data_dir / "image_manifest.csv",
        "images": data_dir / "images",
        "masks": data_dir / "masks",
    }


PATHS = make_drill_data()

train_df = pd.read_csv(PATHS["train_csv"])
test_df = pd.read_csv(PATHS["test_csv"])
sample_submission = pd.read_csv(PATHS["sample_submission_csv"])
records = json.loads(PATHS["records_json"].read_text(encoding="utf-8"))
events = [json.loads(line) for line in PATHS["events_jsonl"].read_text(encoding="utf-8").splitlines()]
M = np.load(PATHS["matrix_npy"])
npz = np.load(PATHS["arrays_npz"])
vecs = npz["vecs"]
texts_df = pd.read_csv(PATHS["texts_csv"])
image_manifest = pd.read_csv(PATHS["image_manifest_csv"])
first_image_path = PATHS["images"] / image_manifest.loc[0, "image_id"]
first_mask_path = PATHS["masks"] / image_manifest.loc[0, "mask_id"]
first_image = Image.open(first_image_path).convert("RGB")
first_mask = Image.open(first_mask_path).convert("L")
image_arr = np.array(first_image)
mask_arr = np.array(first_mask)


def load_image_manifest(data_dir: Path = DATA_DIR) -> pd.DataFrame:
    df = pd.read_csv(data_dir / "image_manifest.csv")
    df["image_path"] = df["image_id"].map(lambda name: data_dir / "images" / name)
    df["mask_path"] = df["mask_id"].map(lambda name: data_dir / "masks" / name)
    return df


def preview_grid(paths, cols: int = 4, size=(2.5, 2.5)):
    import matplotlib.pyplot as plt

    paths = list(paths)
    rows = math.ceil(len(paths) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * size[0], rows * size[1]))
    axes = np.array(axes).reshape(-1)
    for ax, path in zip(axes, paths):
        ax.imshow(Image.open(path))
        ax.set_title(Path(path).name)
        ax.axis("off")
    for ax in axes[len(paths):]:
        ax.axis("off")
    plt.tight_layout()
    return fig


def simple_tokenize(text: str) -> list[str]:
    return re.findall(r"[a-z]+", text.lower())


def build_vocab(texts: list[str], min_freq: int = 1) -> dict[str, int]:
    counts = Counter(tok for text in texts for tok in simple_tokenize(text))
    vocab = {"<PAD>": 0, "<UNK>": 1}
    for token in sorted(tok for tok, count in counts.items() if count >= min_freq):
        vocab[token] = len(vocab)
    return vocab


def encode_tokens(tokens: list[str], vocab: dict[str, int], max_len: int) -> list[int]:
    ids = [vocab.get(tok, vocab["<UNK>"]) for tok in tokens[:max_len]]
    return ids + [vocab["<PAD>"]] * (max_len - len(ids))


def image_to_chw_tensor(img: Image.Image):
    arr = np.asarray(img.convert("RGB"), dtype=np.float32) / 255.0
    chw = np.transpose(arr, (2, 0, 1))
    if torch is None:
        return chw
    return torch.tensor(chw, dtype=torch.float32)


def flip_bbox_horizontal(bbox, width: int):
    x1, y1, x2, y2 = bbox
    return (width - x2 - 1, y1, width - x1 - 1, y2)


def crop_bbox(bbox, crop):
    x1, y1, x2, y2 = bbox
    cx1, cy1, cx2, cy2 = crop
    nx1 = max(0, x1 - cx1)
    ny1 = max(0, y1 - cy1)
    nx2 = min(cx2 - cx1 - 1, x2 - cx1)
    ny2 = min(cy2 - cy1 - 1, y2 - cy1)
    return (nx1, ny1, nx2, ny2)


def normalize_rows(x: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    denom = np.linalg.norm(x, axis=1, keepdims=True)
    return x / np.maximum(denom, eps)


def _same(actual, expected) -> tuple[bool, str]:
    if callable(expected):
        try:
            return bool(expected(actual)), "custom predicate"
        except Exception as exc:
            return False, f"predicate raised {type(exc).__name__}: {exc}"
    if isinstance(actual, Image.Image) and isinstance(expected, Image.Image):
        ok = actual.mode == expected.mode and actual.size == expected.size and np.array_equal(np.asarray(actual), np.asarray(expected))
        return ok, f"expected PIL image mode={expected.mode} size={expected.size}"
    if isinstance(actual, tuple) and isinstance(expected, tuple):
        if len(actual) != len(expected):
            return False, f"tuple length {len(actual)}, expected {len(expected)}"
        results = [_same(a, e)[0] for a, e in zip(actual, expected)]
        return all(results), f"tuple element matches {sum(results)}/{len(results)}"
    if isinstance(actual, list) and isinstance(expected, list):
        if len(actual) != len(expected):
            return False, f"list length {len(actual)}, expected {len(expected)}"
        results = [_same(a, e)[0] for a, e in zip(actual, expected)]
        return all(results), f"list element matches {sum(results)}/{len(results)}"
    if torch is not None and isinstance(actual, torch.Tensor):
        actual = actual.detach().cpu().numpy()
    if torch is not None and isinstance(expected, torch.Tensor):
        expected = expected.detach().cpu().numpy()
    if isinstance(actual, pd.DataFrame) and isinstance(expected, pd.DataFrame):
        return actual.reset_index(drop=True).equals(expected.reset_index(drop=True)), "dataframe equality"
    if isinstance(actual, pd.Series) and isinstance(expected, pd.Series):
        return actual.reset_index(drop=True).equals(expected.reset_index(drop=True)), "series equality"
    if isinstance(actual, np.ndarray) or isinstance(expected, np.ndarray):
        try:
            return np.allclose(np.asarray(actual), np.asarray(expected), equal_nan=True), f"expected array shape {np.asarray(expected).shape}"
        except Exception as exc:
            return False, f"array compare failed: {exc}"
    return actual == expected, f"expected {expected!r}"


def check(cid: str, answer):
    if answer is None:
        print(f"{cid}: TODO - replace answer = None.")
        return False
    if cid not in _EXPECTED:
        print(f"{cid}: no check registered.")
        return False
    ok, detail = _same(answer, _EXPECTED[cid])
    print(f"{cid}: {'PASS' if ok else 'CHECK'} - {detail}")
    return ok


def check_shape(name: str, value, expected_shape):
    shape = tuple(value.shape)
    ok = shape == tuple(expected_shape)
    print(f"{name}: {'PASS' if ok else 'CHECK'} - shape {shape}, expected {expected_shape}")
    return ok


def check_columns(name: str, df: pd.DataFrame, expected_columns: list[str]):
    missing = [col for col in expected_columns if col not in df.columns]
    ok = not missing
    print(f"{name}: {'PASS' if ok else 'CHECK'} - missing columns {missing}")
    return ok


def show_contract(name: str, value) -> None:
    if isinstance(value, pd.DataFrame):
        print(name, "DataFrame", value.shape, list(value.columns))
    elif isinstance(value, np.ndarray):
        print(name, "ndarray", value.shape, value.dtype)
    elif torch is not None and isinstance(value, torch.Tensor):
        print(name, "Tensor", tuple(value.shape), value.dtype, value.device)
    elif isinstance(value, Image.Image):
        print(name, "PIL.Image", value.mode, value.size)
    else:
        print(name, type(value).__name__, repr(value)[:120])


_EXPECTED = {}
_EXPECTED["c001"] = len(records)
_EXPECTED["c002"] = records[0]["id"]
_EXPECTED["c003"] = [r["id"] for r in records if r["label"] == "cat"]
_EXPECTED["c004"] = dict(Counter(r["label"] for r in records))
_EXPECTED["c005"] = max(r["meta"]["score"] for r in records)
_EXPECTED["c006"] = [r["id"] for r in records if "small" in r.get("tags", [])]
_EXPECTED["c007"] = {r["id"]: tuple(r["meta"]["size"]) for r in records}
_EXPECTED["c008"] = defaultdict(list)
for r in records:
    _EXPECTED["c008"][r["label"]].append(r["id"])
_EXPECTED["c008"] = dict(_EXPECTED["c008"])
_EXPECTED["c009"] = [e for e in events if e["event"] == "click"]
_EXPECTED["c010"] = dict(Counter(e["user"] for e in events))
_EXPECTED["c011"] = {u: sum(e["value"] for e in events if e["user"] == u) for u in sorted({e["user"] for e in events})}
_EXPECTED["c012"] = sorted({tag for r in records for tag in r.get("tags", [])})
_EXPECTED["c013"] = [{"id": r["id"], "area": r["meta"]["size"][0] * r["meta"]["size"][1]} for r in records]
_EXPECTED["c014"] = [r.get("confidence", 0.0) for r in records]
_EXPECTED["c015"] = sorted(records, key=lambda r: r["meta"]["score"], reverse=True)[:2]

_EXPECTED["c016"] = M.shape
_EXPECTED["c017"] = M[:, 0]
_EXPECTED["c018"] = M[-2:, -3:]
_EXPECTED["c019"] = M[::-1, ::2]
_EXPECTED["c020"] = M[M % 2 == 0]
_EXPECTED["c021"] = np.where(M > 10)
_EXPECTED["c022"] = np.where(M > 10, 1, 0)
_EXPECTED["c023"] = M.mean(axis=0)
_EXPECTED["c024"] = M.mean(axis=1)
_EXPECTED["c025"] = M - M.mean(axis=0, keepdims=True)
_EXPECTED["c026"] = (M - M.mean(axis=0, keepdims=True)) / M.std(axis=0, keepdims=True)
_EXPECTED["c027"] = np.concatenate([M, M], axis=0)
_EXPECTED["c028"] = np.stack([M[0], M[1]], axis=0)
_EXPECTED["c029"] = M.astype(np.float32)
_EXPECTED["c030"] = M.reshape(2, 3, 4)
_EXPECTED["c031"] = np.linalg.norm(vecs, axis=1)
_EXPECTED["c032"] = normalize_rows(vecs)
_EXPECTED["c033"] = normalize_rows(vecs) @ normalize_rows(vecs).T
_EXPECTED["c034"] = np.clip(vecs, 0, 5)
_EXPECTED["c035"] = np.argsort(M.sum(axis=1))
_EXPECTED["c036"] = M[:, [1, 3, 5]]
_EXPECTED["c037"] = M[[0, 2], [1, 4]]
_EXPECTED["c038"] = np.pad(M, ((1, 1), (2, 0)), constant_values=-1)
_EXPECTED["c039"] = np.load(PATHS["matrix_npy"])
_EXPECTED["c040"] = npz["vecs"]

_EXPECTED["c041"] = train_df.shape
_EXPECTED["c042"] = list(train_df.columns)
_EXPECTED["c043"] = train_df.isna().sum().to_dict()
_EXPECTED["c044"] = train_df["label"]
_EXPECTED["c045"] = train_df.drop(columns=["label"])
_EXPECTED["c046"] = train_df["score"].fillna(train_df["score"].median())
_EXPECTED["c047"] = train_df.assign(score_missing=train_df["score"].isna().astype(int))["score_missing"]
_EXPECTED["c048"] = train_df.groupby("team")["score"].mean()
_EXPECTED["c049"] = train_df.groupby("team")["score"].transform("mean")
_EXPECTED["c050"] = train_df["score"] - train_df.groupby("team")["score"].transform("mean")
_EXPECTED["c051"] = train_df.assign(hours_per_age=train_df["hours"] / train_df["age"])["hours_per_age"]
_EXPECTED["c052"] = pd.get_dummies(train_df["device"], prefix="device", dtype=int)
_EXPECTED["c053"] = train_df.sort_values(["team", "score"], ascending=[True, False])["sample_id"].tolist()
team_counts = train_df["team"].value_counts().rename_axis("team").reset_index(name="team_count")
_EXPECTED["c054"] = train_df.merge(team_counts, on="team", validate="many_to_one")
_EXPECTED["c055"] = train_df.query("score >= 80")["sample_id"].tolist()
_EXPECTED["c056"] = train_df.loc[train_df["height_cm"] > 200, "sample_id"].tolist()
_EXPECTED["c057"] = train_df[["score", "hours", "age"]].corr()
_EXPECTED["c058"] = train_df.groupby("team").agg(score_mean=("score", "mean"), hours_sum=("hours", "sum"))
_EXPECTED["c059"] = sample_submission.shape
_EXPECTED["c060"] = list(sample_submission.columns)

_EXPECTED["c061"] = PATHS["train_csv"].exists()
_EXPECTED["c062"] = sorted(p.suffix for p in DATA_DIR.rglob("*") if p.is_file())
_EXPECTED["c063"] = json.loads(PATHS["records_json"].read_text(encoding="utf-8"))
_EXPECTED["c064"] = events
_EXPECTED["c065"] = pd.read_json(PATHS["events_jsonl"], lines=True)
_EXPECTED["c066"] = np.load(PATHS["matrix_npy"]).shape
_EXPECTED["c067"] = sorted(np.load(PATHS["arrays_npz"]).files)
_EXPECTED["c068"] = pd.read_csv(PATHS["train_csv"]).equals(train_df)

if torch is not None:
    _EXPECTED["c069"] = torch.tensor(vecs, dtype=torch.float32)
    _EXPECTED["c070"] = torch.tensor(train_df["label"].values, dtype=torch.long)
    _EXPECTED["c071"] = torch.tensor(train_df[["age", "height_cm", "hours"]].values, dtype=torch.float32)
    _EXPECTED["c072"] = torch.tensor(train_df[["age", "height_cm", "hours"]].values, dtype=torch.float32).shape
    _EXPECTED["c073"] = torch.tensor(train_df[["age", "height_cm", "hours"]].values, dtype=torch.float32)[:4]
    _EXPECTED["c074"] = torch.tensor(train_df["score"].fillna(0).values, dtype=torch.float32) > 70
    _EXPECTED["c075"] = F.one_hot(torch.tensor(train_df["label"].values, dtype=torch.long), num_classes=2)
    feats_t = torch.tensor(train_df[["age", "height_cm", "hours"]].values, dtype=torch.float32)
    _EXPECTED["c076"] = (feats_t - feats_t.mean(dim=0, keepdim=True)) / feats_t.std(dim=0, keepdim=True)
    _EXPECTED["c077"] = torch.linalg.vector_norm(torch.tensor(vecs, dtype=torch.float32), dim=1)
    _EXPECTED["c078"] = F.normalize(torch.tensor(vecs, dtype=torch.float32), p=2, dim=1)
    _EXPECTED["c079"] = torch.stack([image_to_chw_tensor(first_image), image_to_chw_tensor(Image.open(PATHS["images"] / image_manifest.loc[1, "image_id"]))])
    _EXPECTED["c080"] = tuple(_EXPECTED["c079"].shape)
else:
    for cid in range(69, 81):
        _EXPECTED[f"c{cid:03d}"] = lambda answer: False

vocab = build_vocab(texts_df["text"].tolist())
first_tokens = simple_tokenize(texts_df.loc[0, "text"])
_EXPECTED["c081"] = first_tokens
_EXPECTED["c082"] = vocab
_EXPECTED["c083"] = encode_tokens(first_tokens, vocab, max_len=6)
_EXPECTED["c084"] = [len(simple_tokenize(t)) for t in texts_df["text"]]
_EXPECTED["c085"] = sorted(set(tok for text in texts_df["text"] for tok in simple_tokenize(text)))
_EXPECTED["c086"] = Counter(tok for text in texts_df["text"] for tok in simple_tokenize(text))
_EXPECTED["c087"] = np.array([[text.lower().count(tok) for tok in ["cats", "dogs", "birds", "outside"]] for text in texts_df["text"]])
_EXPECTED["c088"] = texts_df["text"].str.lower().str.replace(r"[^a-z ]", "", regex=True)
_EXPECTED["c089"] = texts_df.assign(n_tokens=[len(simple_tokenize(t)) for t in texts_df["text"]])["n_tokens"]
_EXPECTED["c090"] = np.array([encode_tokens(simple_tokenize(t), vocab, max_len=5) for t in texts_df["text"]])

_EXPECTED["c091"] = first_image.mode
_EXPECTED["c092"] = first_image.size
_EXPECTED["c093"] = first_image.resize((28, 28)).size
_EXPECTED["c094"] = first_image.crop((8, 8, 32, 32)).size
_EXPECTED["c095"] = ImageOps.grayscale(first_image).mode
_EXPECTED["c096"] = np.asarray(first_image).shape
_EXPECTED["c097"] = np.asarray(first_image).astype(np.float32) / 255.0
_EXPECTED["c098"] = np.transpose(np.asarray(first_image).astype(np.float32) / 255.0, (2, 0, 1))
_EXPECTED["c099"] = tuple(_EXPECTED["c098"].shape)
_EXPECTED["c100"] = np.asarray(first_mask).shape
_EXPECTED["c101"] = (np.asarray(first_mask) > 0).astype(np.uint8)
_EXPECTED["c102"] = load_image_manifest().shape

_EXPECTED["c103"] = ImageOps.mirror(first_image)
_EXPECTED["c104"] = ImageOps.flip(first_image)
_EXPECTED["c105"] = first_image.rotate(15, resample=Image.Resampling.BILINEAR)
_EXPECTED["c106"] = ImageOps.pad(first_image, (64, 64), color=(0, 0, 0))
_EXPECTED["c107"] = first_image.crop((4, 4, 44, 44))
_EXPECTED["c108"] = first_image.crop((12, 12, 44, 44)).resize((56, 56))
_EXPECTED["c109"] = ImageEnhance.Brightness(first_image).enhance(1.5)
_EXPECTED["c110"] = ImageEnhance.Contrast(first_image).enhance(1.4)
_EXPECTED["c111"] = first_image.filter(ImageFilter.GaussianBlur(radius=1.2))
rng_noise = np.random.default_rng(123)
noise = rng_noise.normal(0, 10, size=np.asarray(first_image).shape)
_EXPECTED["c112"] = np.clip(np.asarray(first_image).astype(np.float32) + noise, 0, 255).astype(np.uint8)
_EXPECTED["c113"] = (np.asarray(ImageOps.grayscale(first_image)) > 40).astype(np.uint8)
_EXPECTED["c114"] = ImageOps.mirror(first_mask)
bbox0 = tuple(image_manifest.loc[0, ["x1", "y1", "x2", "y2"]].astype(int))
_EXPECTED["c115"] = flip_bbox_horizontal(bbox0, width=first_image.width)
_EXPECTED["c116"] = crop_bbox(bbox0, crop=(4, 4, 44, 44))
_EXPECTED["c117"] = (ImageOps.mirror(first_image), ImageOps.mirror(first_mask), flip_bbox_horizontal(bbox0, first_image.width))
_EXPECTED["c118"] = lambda ans: isinstance(ans, list) and len(ans) == 4 and all(isinstance(img, Image.Image) for img in ans)

print("Ready.")
print("Data directory:", DATA_DIR.resolve())
print("Registered checks:", len(_EXPECTED))
show_contract("train_df", train_df)
show_contract("M", M)
show_contract("first_image", first_image)


Ready.
Data directory: D:\projects\Supervised-Learning-Experiments\olympiads\IOAI Material\1. Basics\exercises\_drill_data
Registered checks: 118
train_df DataFrame (12, 8) ['sample_id', 'team', 'age', 'height_cm', 'score', 'hours', 'device', 'label']
M ndarray (4, 6) int64
first_image PIL.Image RGB (56, 56)


## Drill Rules

- Do not edit the setup/check helpers.
- Prefer one clean expression when possible.
- Do not use model code until the data contract is clear.
- For image drills, always track mode, size, dtype, and `HWC` vs `CHW`.


## Python Records, Dictionaries, JSONL

Contest datasets often start as nested records. These drills make record inspection automatic.


In [2]:
# Challenge 001 - record count (1 min)
# Task: Return the number of records in `records`.
# Expected contract: integer
answer = len(records)
check("c001", answer)


c001: PASS - expected 5


True

In [130]:
# Challenge 002 - first id (1 min)
# Task: Return the `id` of the first record.
# Expected contract: string
# print(records)
answer = records[0]["id"]
check("c002", answer)


c002: PASS - expected 'r001'


True

In [133]:
# Challenge 003 - filter labels (2 min)
# Task: Return IDs for records whose label is `cat`.
# Expected contract: list of strings
answer = [i["id"] for i in records if i["label"]=="cat"]
check("c003", answer)


c003: PASS - list element matches 2/2


True

In [5]:
# Challenge 004 - count labels (2 min)
# Task: Return label counts as a plain dict.
# Expected contract: dict label -> count
answer = None
check("c004", answer)


c004: TODO - replace answer = None.


False

In [6]:
# Challenge 005 - nested max (2 min)
# Task: Return the maximum nested `meta.score`.
# Expected contract: float
answer = None
check("c005", answer)


c005: TODO - replace answer = None.


False

In [7]:
# Challenge 006 - tag filter (2 min)
# Task: Return IDs for records containing tag `small`.
# Expected contract: list of strings
answer = None
check("c006", answer)


c006: TODO - replace answer = None.


False

In [8]:
# Challenge 007 - id to size (2 min)
# Task: Return a dict mapping record id to size tuple.
# Expected contract: dict id -> tuple
answer = None
check("c007", answer)


c007: TODO - replace answer = None.


False

In [9]:
# Challenge 008 - group ids (3 min)
# Task: Group record IDs by label.
# Expected contract: dict label -> list of IDs
answer = None
check("c008", answer)


c008: TODO - replace answer = None.


False

In [10]:
# Challenge 009 - jsonl filter (2 min)
# Task: Return event dicts whose event is `click`.
# Expected contract: list of dicts
answer = None
check("c009", answer)


c009: TODO - replace answer = None.


False

In [11]:
# Challenge 010 - user event counts (2 min)
# Task: Return event counts per user.
# Expected contract: dict user -> count
answer = None
check("c010", answer)


c010: TODO - replace answer = None.


False

In [12]:
# Challenge 011 - sum values (3 min)
# Task: Return total event value per user with sorted user keys.
# Expected contract: dict user -> int
answer = None
check("c011", answer)


c011: TODO - replace answer = None.


False

In [13]:
# Challenge 012 - unique tags (2 min)
# Task: Return sorted unique tags across all records.
# Expected contract: sorted list
answer = None
check("c012", answer)


c012: TODO - replace answer = None.


False

In [14]:
# Challenge 013 - derived area (3 min)
# Task: Return a list of dicts with `id` and pixel `area`.
# Expected contract: list of dicts
answer = None
check("c013", answer)


c013: TODO - replace answer = None.


False

In [15]:
# Challenge 014 - safe missing key (1 min)
# Task: Return each record's missing `confidence`, defaulting to `0.0`.
# Expected contract: list of floats
answer = None
check("c014", answer)


c014: TODO - replace answer = None.


False

In [16]:
# Challenge 015 - sort by nested score (3 min)
# Task: Return the top 2 records by descending `meta.score`.
# Expected contract: list of 2 dicts
answer = None
check("c015", answer)


c015: TODO - replace answer = None.


False

## NumPy Arrays, Normalization, Similarity

This block targets shapes, vector normalization, broadcasting, and disk array loading.


In [17]:
# Challenge 016 - matrix shape (30 sec)
# Task: Return `M`'s shape.
# Expected contract: tuple
answer = None
check("c016", answer)


c016: TODO - replace answer = None.


False

In [18]:
# Challenge 017 - first column (1 min)
# Task: Return the first column of `M`.
# Expected contract: array shape (4,)
answer = None
check("c017", answer)


c017: TODO - replace answer = None.


False

In [19]:
# Challenge 018 - bottom-right slice (1 min)
# Task: Return the last 2 rows and last 3 columns of `M`.
# Expected contract: array shape (2, 3)
answer = None
check("c018", answer)


c018: TODO - replace answer = None.


False

In [20]:
# Challenge 019 - reverse step slice (2 min)
# Task: Reverse row order and take every other column from `M`.
# Expected contract: array shape (4, 3)
answer = None
check("c019", answer)


c019: TODO - replace answer = None.


False

In [21]:
# Challenge 020 - even mask values (2 min)
# Task: Return all even values in `M`.
# Expected contract: 1D array
answer = None
check("c020", answer)


c020: TODO - replace answer = None.


False

In [22]:
# Challenge 021 - where coordinates (2 min)
# Task: Return `np.where` coordinates where `M > 10`.
# Expected contract: tuple of index arrays
answer = None
check("c021", answer)


c021: TODO - replace answer = None.


False

In [23]:
# Challenge 022 - binary threshold (2 min)
# Task: Return a 0/1 array where `M > 10`.
# Expected contract: array shape (4, 6)
answer = None
check("c022", answer)


c022: TODO - replace answer = None.


False

In [24]:
# Challenge 023 - column means (1 min)
# Task: Return mean of each column.
# Expected contract: array shape (6,)
answer = None
check("c023", answer)


c023: TODO - replace answer = None.


False

In [25]:
# Challenge 024 - row means (1 min)
# Task: Return mean of each row.
# Expected contract: array shape (4,)
answer = None
check("c024", answer)


c024: TODO - replace answer = None.


False

In [26]:
# Challenge 025 - center columns (2 min)
# Task: Subtract each column mean from `M`.
# Expected contract: array shape (4, 6)
answer = None
check("c025", answer)


c025: TODO - replace answer = None.


False

In [27]:
# Challenge 026 - standardize columns (3 min)
# Task: Z-score each column of `M`.
# Expected contract: array shape (4, 6)
answer = None
check("c026", answer)


c026: TODO - replace answer = None.


False

In [28]:
# Challenge 027 - concat rows (1 min)
# Task: Concatenate `M` with itself by rows.
# Expected contract: array shape (8, 6)
answer = None
check("c027", answer)


c027: TODO - replace answer = None.


False

In [29]:
# Challenge 028 - stack rows (1 min)
# Task: Stack the first two rows of `M` into a new array.
# Expected contract: array shape (2, 6)
answer = None
check("c028", answer)


c028: TODO - replace answer = None.


False

In [30]:
# Challenge 029 - astype float32 (1 min)
# Task: Convert `M` to `float32`.
# Expected contract: float32 array
answer = None
check("c029", answer)


c029: TODO - replace answer = None.


False

In [31]:
# Challenge 030 - reshape 3d (1 min)
# Task: Reshape `M` into `(2, 3, 4)`.
# Expected contract: array shape (2, 3, 4)
answer = None
check("c030", answer)


c030: TODO - replace answer = None.


False

In [32]:
# Challenge 031 - vector norms (2 min)
# Task: Return L2 norm for each row in `vecs`.
# Expected contract: array shape (4,)
answer = None
check("c031", answer)


c031: TODO - replace answer = None.


False

In [33]:
# Challenge 032 - row normalize (3 min)
# Task: Return row-wise L2-normalized `vecs`, keeping zero rows stable.
# Expected contract: array shape (4, 2)
answer = None
check("c032", answer)


c032: TODO - replace answer = None.


False

In [34]:
# Challenge 033 - cosine matrix (4 min)
# Task: Return cosine similarity matrix for rows of `vecs`.
# Expected contract: array shape (4, 4)
answer = None
check("c033", answer)


c033: TODO - replace answer = None.


False

In [35]:
# Challenge 034 - clip values (1 min)
# Task: Clip `vecs` to range `[0, 5]`.
# Expected contract: array shape (4, 2)
answer = None
check("c034", answer)


c034: TODO - replace answer = None.


False

In [36]:
# Challenge 035 - argsort rows (2 min)
# Task: Return row indices sorted by row sum of `M`.
# Expected contract: array shape (4,)
answer = None
check("c035", answer)


c035: TODO - replace answer = None.


False

In [37]:
# Challenge 036 - fancy columns (1 min)
# Task: Select columns 1, 3, and 5 from `M`.
# Expected contract: array shape (4, 3)
answer = None
check("c036", answer)


c036: TODO - replace answer = None.


False

In [38]:
# Challenge 037 - paired indexing (2 min)
# Task: Return values at coordinate pairs `(0,1)` and `(2,4)`.
# Expected contract: array shape (2,)
answer = None
check("c037", answer)


c037: TODO - replace answer = None.


False

In [39]:
# Challenge 038 - pad matrix (3 min)
# Task: Pad `M` with 1 row above/below and 2 columns on the left using `-1`.
# Expected contract: array shape (6, 8)
answer = None
check("c038", answer)


c038: TODO - replace answer = None.


False

In [40]:
# Challenge 039 - load npy (1 min)
# Task: Load `PATHS['matrix_npy']` from disk.
# Expected contract: array shape (4, 6)
answer = None
check("c039", answer)


c039: TODO - replace answer = None.


False

In [41]:
# Challenge 040 - load npz vecs (1 min)
# Task: Read `vecs` from the `.npz` file.
# Expected contract: array shape (4, 2)
answer = None
check("c040", answer)


c040: TODO - replace answer = None.


False

## Pandas, CSV Contracts, Feature Tables

These drills train the checks that prevent baseline bugs: columns, missingness, merges, leakage, and submissions.


In [42]:
# Challenge 041 - csv shape (1 min)
# Task: Return shape of `train_df`.
# Expected contract: tuple
answer = None
check("c041", answer)


c041: TODO - replace answer = None.


False

In [43]:
# Challenge 042 - csv columns (1 min)
# Task: Return column names as a list.
# Expected contract: list of strings
answer = None
check("c042", answer)


c042: TODO - replace answer = None.


False

In [44]:
# Challenge 043 - missing counts (2 min)
# Task: Return missing count per column as a dict.
# Expected contract: dict
answer = None
check("c043", answer)


c043: TODO - replace answer = None.


False

In [45]:
# Challenge 044 - target series (1 min)
# Task: Return the target label series.
# Expected contract: Series length 12
answer = None
check("c044", answer)


c044: TODO - replace answer = None.


False

In [46]:
# Challenge 045 - drop target (1 min)
# Task: Return features after dropping `label`.
# Expected contract: DataFrame with no label column
answer = None
check("c045", answer)


c045: TODO - replace answer = None.


False

In [47]:
# Challenge 046 - fill score median (2 min)
# Task: Return `score` filled with the median score.
# Expected contract: Series length 12
answer = None
check("c046", answer)


c046: TODO - replace answer = None.


False

In [48]:
# Challenge 047 - missing indicator (2 min)
# Task: Return a 0/1 `score_missing` series.
# Expected contract: Series length 12
answer = None
check("c047", answer)


c047: TODO - replace answer = None.


False

In [49]:
# Challenge 048 - group mean (2 min)
# Task: Return mean score by team.
# Expected contract: Series indexed by team
answer = None
check("c048", answer)


c048: TODO - replace answer = None.


False

In [50]:
# Challenge 049 - group transform (2 min)
# Task: Return team mean score aligned to original rows.
# Expected contract: Series length 12
answer = None
check("c049", answer)


c049: TODO - replace answer = None.


False

In [51]:
# Challenge 050 - group-relative feature (3 min)
# Task: Return score minus team mean score.
# Expected contract: Series length 12
answer = None
check("c050", answer)


c050: TODO - replace answer = None.


False

In [52]:
# Challenge 051 - ratio feature (2 min)
# Task: Return hours divided by age.
# Expected contract: Series length 12
answer = None
check("c051", answer)


c051: TODO - replace answer = None.


False

In [53]:
# Challenge 052 - one hot device (3 min)
# Task: One-hot encode the `device` column with prefix `device`.
# Expected contract: DataFrame
answer = None
check("c052", answer)


c052: TODO - replace answer = None.


False

In [54]:
# Challenge 053 - sort sample ids (2 min)
# Task: Return sample IDs sorted by team then descending score.
# Expected contract: list
answer = None
check("c053", answer)


c053: TODO - replace answer = None.


False

In [55]:
# Challenge 054 - safe merge (3 min)
# Task: Merge team counts back to `train_df` with `validate='many_to_one'`.
# Expected contract: DataFrame
answer = None
check("c054", answer)


c054: TODO - replace answer = None.


False

In [56]:
# Challenge 055 - high score ids (1 min)
# Task: Return sample IDs with score at least 80.
# Expected contract: list
answer = None
check("c055", answer)


c055: TODO - replace answer = None.


False

In [57]:
# Challenge 056 - outlier ids (1 min)
# Task: Return sample IDs with height over 200.
# Expected contract: list
answer = None
check("c056", answer)


c056: TODO - replace answer = None.


False

In [58]:
# Challenge 057 - correlation (2 min)
# Task: Return correlation table for score, hours, and age.
# Expected contract: DataFrame
answer = None
check("c057", answer)


c057: TODO - replace answer = None.


False

In [59]:
# Challenge 058 - aggregate table (3 min)
# Task: Return team-level score mean and hours sum using named aggregation.
# Expected contract: DataFrame
answer = None
check("c058", answer)


c058: TODO - replace answer = None.


False

In [60]:
# Challenge 059 - sample submission shape (1 min)
# Task: Return `sample_submission` shape.
# Expected contract: tuple
answer = None
check("c059", answer)


c059: TODO - replace answer = None.


False

In [61]:
# Challenge 060 - sample submission columns (1 min)
# Task: Return sample submission columns.
# Expected contract: list
answer = None
check("c060", answer)


c060: TODO - replace answer = None.


False

## File Discovery And Disk Reloads

A prediction file does not count until it reloads cleanly from disk.


In [62]:
# Challenge 061 - path exists (30 sec)
# Task: Return whether the train CSV path exists.
# Expected contract: bool
answer = None
check("c061", answer)


c061: TODO - replace answer = None.


False

In [63]:
# Challenge 062 - suffix inventory (2 min)
# Task: Return sorted suffixes for all files under `DATA_DIR`.
# Expected contract: list
answer = None
check("c062", answer)


c062: TODO - replace answer = None.


False

In [64]:
# Challenge 063 - read json (1 min)
# Task: Read `records.json` from disk.
# Expected contract: list of records
answer = None
check("c063", answer)


c063: TODO - replace answer = None.


False

In [65]:
# Challenge 064 - read jsonl (2 min)
# Task: Read `events.jsonl` into a list of dicts.
# Expected contract: list of dicts
answer = None
check("c064", answer)


c064: TODO - replace answer = None.


False

In [66]:
# Challenge 065 - jsonl dataframe (2 min)
# Task: Read `events.jsonl` into a DataFrame.
# Expected contract: DataFrame
answer = None
check("c065", answer)


c065: TODO - replace answer = None.


False

In [67]:
# Challenge 066 - npy shape (1 min)
# Task: Load the `.npy` file and return its shape.
# Expected contract: tuple
answer = None
check("c066", answer)


c066: TODO - replace answer = None.


False

In [68]:
# Challenge 067 - npz keys (1 min)
# Task: Return sorted keys inside the `.npz` file.
# Expected contract: list
answer = None
check("c067", answer)


c067: TODO - replace answer = None.


False

In [69]:
# Challenge 068 - csv reload equality (2 min)
# Task: Reload train CSV and check equality with `train_df`.
# Expected contract: bool
answer = None
check("c068", answer)


c068: TODO - replace answer = None.


False

## PyTorch Tensors And Batch Contracts

These drills make dtype and shape checks automatic before training.


In [70]:
# Challenge 069 - numpy to tensor (1 min)
# Task: Convert `vecs` to a float32 tensor.
# Expected contract: tensor shape (4, 2)
answer = None
check("c069", answer)


c069: TODO - replace answer = None.


False

In [71]:
# Challenge 070 - labels tensor (1 min)
# Task: Convert labels to a long tensor.
# Expected contract: tensor shape (12,)
answer = None
check("c070", answer)


c070: TODO - replace answer = None.


False

In [72]:
# Challenge 071 - feature tensor (2 min)
# Task: Convert age, height, hours columns to a float32 tensor.
# Expected contract: tensor shape (12, 3)
answer = None
check("c071", answer)


c071: TODO - replace answer = None.


False

In [73]:
# Challenge 072 - tensor shape (30 sec)
# Task: Return the shape tuple of that feature tensor.
# Expected contract: tuple
answer = None
check("c072", answer)


c072: TODO - replace answer = None.


False

In [74]:
# Challenge 073 - first batch (1 min)
# Task: Return the first 4 rows of the feature tensor.
# Expected contract: tensor shape (4, 3)
answer = None
check("c073", answer)


c073: TODO - replace answer = None.


False

In [75]:
# Challenge 074 - score mask (2 min)
# Task: Return a boolean tensor for filled score greater than 70.
# Expected contract: bool tensor
answer = None
check("c074", answer)


c074: TODO - replace answer = None.


False

In [76]:
# Challenge 075 - one hot labels (2 min)
# Task: One-hot encode labels into 2 classes.
# Expected contract: tensor shape (12, 2)
answer = None
check("c075", answer)


c075: TODO - replace answer = None.


False

In [77]:
# Challenge 076 - standardize tensor (3 min)
# Task: Standardize feature tensor column-wise.
# Expected contract: tensor shape (12, 3)
answer = None
check("c076", answer)


c076: TODO - replace answer = None.


False

In [78]:
# Challenge 077 - tensor vector norms (2 min)
# Task: Return L2 norm for each row of `vecs` as a tensor.
# Expected contract: tensor shape (4,)
answer = None
check("c077", answer)


c077: TODO - replace answer = None.


False

In [79]:
# Challenge 078 - tensor normalize (2 min)
# Task: L2-normalize rows of `vecs` using PyTorch.
# Expected contract: tensor shape (4, 2)
answer = None
check("c078", answer)


c078: TODO - replace answer = None.


False

In [80]:
# Challenge 079 - image tensor batch (3 min)
# Task: Stack tensors for the first two manifest images.
# Expected contract: tensor shape (2, 3, 56, 56)
answer = None
check("c079", answer)


c079: TODO - replace answer = None.


False

In [81]:
# Challenge 080 - image batch shape (30 sec)
# Task: Return the shape tuple of that image batch.
# Expected contract: tuple
answer = None
check("c080", answer)


c080: TODO - replace answer = None.


False

## Tokenization And Text Features

This block trains simple NLP preprocessing without hiding the data contract.


In [82]:
# Challenge 081 - basic tokenize (1 min)
# Task: Tokenize the first text using `simple_tokenize`.
# Expected contract: list of tokens
answer = None
check("c081", answer)


c081: TODO - replace answer = None.


False

In [83]:
# Challenge 082 - build vocab (3 min)
# Task: Build a vocabulary from all texts.
# Expected contract: dict token -> id
answer = None
check("c082", answer)


c082: TODO - replace answer = None.


False

In [84]:
# Challenge 083 - encode padded (2 min)
# Task: Encode first text to length 6 using the vocab.
# Expected contract: list of 6 ints
answer = None
check("c083", answer)


c083: TODO - replace answer = None.


False

In [85]:
# Challenge 084 - token counts (2 min)
# Task: Return token count per text.
# Expected contract: list of ints
answer = None
check("c084", answer)


c084: TODO - replace answer = None.


False

In [86]:
# Challenge 085 - unique tokens (2 min)
# Task: Return sorted unique tokens across all texts.
# Expected contract: list
answer = None
check("c085", answer)


c085: TODO - replace answer = None.


False

In [87]:
# Challenge 086 - token frequency (3 min)
# Task: Return token frequency Counter across all texts.
# Expected contract: Counter
answer = None
check("c086", answer)


c086: TODO - replace answer = None.


False

In [88]:
# Challenge 087 - manual bow (4 min)
# Task: Build counts for tokens cats/dogs/birds/outside per text.
# Expected contract: array shape (6, 4)
answer = None
check("c087", answer)


c087: TODO - replace answer = None.


False

In [89]:
# Challenge 088 - clean text series (2 min)
# Task: Lowercase text and remove punctuation.
# Expected contract: Series
answer = None
check("c088", answer)


c088: TODO - replace answer = None.


False

In [90]:
# Challenge 089 - n_tokens column (2 min)
# Task: Add token counts and return that column.
# Expected contract: Series
answer = None
check("c089", answer)


c089: TODO - replace answer = None.


False

In [91]:
# Challenge 090 - token id matrix (4 min)
# Task: Encode every text to length 5.
# Expected contract: array shape (6, 5)
answer = None
check("c090", answer)


c090: TODO - replace answer = None.


False

## Computer Vision Loading And Tensor Conversion

Image preprocessing bugs are usually mode, size, dtype, or channel-order bugs.


In [92]:
# Challenge 091 - image mode (30 sec)
# Task: Return `first_image.mode`.
# Expected contract: string
answer = None
check("c091", answer)


c091: TODO - replace answer = None.


False

In [93]:
# Challenge 092 - image size (30 sec)
# Task: Return `first_image.size`.
# Expected contract: tuple width,height
answer = None
check("c092", answer)


c092: TODO - replace answer = None.


False

In [94]:
# Challenge 093 - resize image (1 min)
# Task: Resize `first_image` to `(28, 28)` and return its size.
# Expected contract: tuple
answer = None
check("c093", answer)


c093: TODO - replace answer = None.


False

In [95]:
# Challenge 094 - crop image (1 min)
# Task: Crop box `(8, 8, 32, 32)` and return size.
# Expected contract: tuple
answer = None
check("c094", answer)


c094: TODO - replace answer = None.


False

In [96]:
# Challenge 095 - grayscale mode (1 min)
# Task: Convert image to grayscale and return mode.
# Expected contract: string
answer = None
check("c095", answer)


c095: TODO - replace answer = None.


False

In [97]:
# Challenge 096 - image array shape (1 min)
# Task: Convert first image to NumPy and return shape.
# Expected contract: tuple
answer = None
check("c096", answer)


c096: TODO - replace answer = None.


False

In [98]:
# Challenge 097 - image to 0-1 (2 min)
# Task: Convert first image to float array in `[0, 1]`.
# Expected contract: array shape (56, 56, 3)
answer = None
check("c097", answer)


c097: TODO - replace answer = None.


False

In [99]:
# Challenge 098 - HWC to CHW (2 min)
# Task: Convert normalized image array from HWC to CHW.
# Expected contract: array shape (3, 56, 56)
answer = None
check("c098", answer)


c098: TODO - replace answer = None.


False

In [100]:
# Challenge 099 - CHW shape (30 sec)
# Task: Return CHW shape tuple.
# Expected contract: tuple
answer = None
check("c099", answer)


c099: TODO - replace answer = None.


False

In [101]:
# Challenge 100 - mask shape (1 min)
# Task: Convert first mask to array and return shape.
# Expected contract: tuple
answer = None
check("c100", answer)


c100: TODO - replace answer = None.


False

In [102]:
# Challenge 101 - binary mask (2 min)
# Task: Convert first mask to binary 0/1 array.
# Expected contract: array shape (56, 56)
answer = None
check("c101", answer)


c101: TODO - replace answer = None.


False

In [103]:
# Challenge 102 - manifest shape (1 min)
# Task: Load image manifest with paths and return shape.
# Expected contract: tuple
answer = None
check("c102", answer)


c102: TODO - replace answer = None.


False

## Image Augmentation And Label-Aware CV Contracts

Augmentation is only correct if labels, masks, and boxes still mean the same thing.


In [104]:
# Challenge 103 - horizontal flip (1 min)
# Task: Horizontally flip `first_image`.
# Expected contract: PIL image
answer = None
check("c103", answer)


c103: TODO - replace answer = None.


False

In [105]:
# Challenge 104 - vertical flip (1 min)
# Task: Vertically flip `first_image`.
# Expected contract: PIL image
answer = None
check("c104", answer)


c104: TODO - replace answer = None.


False

In [106]:
# Challenge 105 - small rotation (2 min)
# Task: Rotate `first_image` by 15 degrees with bilinear resampling.
# Expected contract: PIL image
answer = None
check("c105", answer)


c105: TODO - replace answer = None.


False

In [107]:
# Challenge 106 - pad image (2 min)
# Task: Pad `first_image` to `(64, 64)` with black.
# Expected contract: PIL image
answer = None
check("c106", answer)


c106: TODO - replace answer = None.


False

In [108]:
# Challenge 107 - fixed crop (1 min)
# Task: Crop box `(4, 4, 44, 44)`.
# Expected contract: PIL image
answer = None
check("c107", answer)


c107: TODO - replace answer = None.


False

In [109]:
# Challenge 108 - crop then resize (2 min)
# Task: Crop `(12, 12, 44, 44)` then resize back to `(56, 56)`.
# Expected contract: PIL image
answer = None
check("c108", answer)


c108: TODO - replace answer = None.


False

In [110]:
# Challenge 109 - brightness (1 min)
# Task: Increase image brightness by factor 1.5.
# Expected contract: PIL image
answer = None
check("c109", answer)


c109: TODO - replace answer = None.


False

In [111]:
# Challenge 110 - contrast (1 min)
# Task: Increase image contrast by factor 1.4.
# Expected contract: PIL image
answer = None
check("c110", answer)


c110: TODO - replace answer = None.


False

In [112]:
# Challenge 111 - blur (1 min)
# Task: Apply Gaussian blur radius 1.2.
# Expected contract: PIL image
answer = None
check("c111", answer)


c111: TODO - replace answer = None.


False

In [113]:
# Challenge 112 - add noise (3 min)
# Task: Add deterministic Gaussian noise with RNG seed 123, std 10, clipped to uint8.
# Expected contract: uint8 array
answer = None
check("c112", answer)


c112: TODO - replace answer = None.


False

In [114]:
# Challenge 113 - threshold (2 min)
# Task: Grayscale then threshold image at >40 into 0/1 array.
# Expected contract: array shape (56, 56)
answer = None
check("c113", answer)


c113: TODO - replace answer = None.


False

In [115]:
# Challenge 114 - flip mask (1 min)
# Task: Horizontally flip the mask exactly like the image.
# Expected contract: PIL mask
answer = None
check("c114", answer)


c114: TODO - replace answer = None.


False

In [116]:
# Challenge 115 - flip bbox (3 min)
# Task: Horizontally flip `bbox0` for image width 56.
# Expected contract: tuple x1,y1,x2,y2
answer = None
check("c115", answer)


c115: TODO - replace answer = None.


False

In [117]:
# Challenge 116 - crop bbox (3 min)
# Task: Adjust `bbox0` after crop `(4, 4, 44, 44)`.
# Expected contract: tuple
answer = None
check("c116", answer)


c116: TODO - replace answer = None.


False

In [118]:
# Challenge 117 - joint transform tuple (4 min)
# Task: Return flipped image, flipped mask, and flipped bbox together.
# Expected contract: tuple
answer = None
check("c117", answer)


c117: TODO - replace answer = None.


False

In [119]:
# Challenge 118 - augmentation list (5 min)
# Task: Return four PIL image augmentations of `first_image`.
# Expected contract: list of 4 PIL images
answer = None
check("c118", answer)


c118: TODO - replace answer = None.


False

## Model-Use Sanity Checks

These cells intentionally include working model code. The point is to prove that preprocessing produced model-ready objects:

- tabular features become a tree ensemble input;
- text becomes a sparse bag-of-words matrix;
- images become a `NCHW` tensor batch for a tiny CNN.


In [120]:
seed_everything(42)

model_train = train_df.copy()
model_train["score"] = model_train["score"].fillna(model_train["score"].median())
X_tab = pd.get_dummies(model_train[["team", "age", "height_cm", "score", "hours", "device"]], columns=["team", "device"], dtype=int)
y_tab = model_train["label"].astype(int)

if RandomForestClassifier is not None:
    clf = RandomForestClassifier(n_estimators=25, max_depth=3, random_state=42)
    clf.fit(X_tab, y_tab)
    tab_pred = clf.predict(X_tab)
    print("Tabular RandomForest train accuracy:", round(float(accuracy_score(y_tab, tab_pred)), 3))
    show_contract("X_tab", X_tab)
else:
    print("Skipping tabular model because sklearn is unavailable.")

if CountVectorizer is not None:
    vectorizer = CountVectorizer(lowercase=True)
    X_text = vectorizer.fit_transform(texts_df["text"])
    text_clf = LogisticRegression(max_iter=200, random_state=42)
    text_clf.fit(X_text, texts_df["label"])
    text_pred = text_clf.predict(X_text)
    print("Text LogisticRegression train accuracy:", round(float(accuracy_score(texts_df["label"], text_pred)), 3))
    print("Vocabulary size:", len(vectorizer.vocabulary_))
else:
    print("Skipping text model because sklearn is unavailable.")

if torch is not None:
    manifest = load_image_manifest()
    image_tensors = []
    labels = []
    for _, row in manifest.iterrows():
        img = Image.open(row["image_path"]).convert("RGB").resize((32, 32))
        image_tensors.append(image_to_chw_tensor(img))
        labels.append(int(row["label"]))
    X_img = torch.stack(image_tensors)
    y_img = torch.tensor(labels, dtype=torch.long)

    class TinyCNN(nn.Module):
        def __init__(self):
            super().__init__()
            self.conv = nn.Conv2d(3, 8, kernel_size=3, padding=1)
            self.pool = nn.AdaptiveAvgPool2d((1, 1))
            self.head = nn.Linear(8, 2)

        def forward(self, x):
            x = F.relu(self.conv(x))
            x = self.pool(x).flatten(1)
            return self.head(x)

    cnn = TinyCNN()
    opt = torch.optim.Adam(cnn.parameters(), lr=0.01)
    logits = cnn(X_img)
    loss = F.cross_entropy(logits, y_img)
    opt.zero_grad()
    loss.backward()
    opt.step()
    print("Image batch:", tuple(X_img.shape))
    print("CNN logits:", tuple(logits.shape))
    print("One tiny training loss:", round(float(loss.item()), 4))
else:
    print("Skipping tiny CNN because torch is unavailable.")


Tabular RandomForest train accuracy: 1.0
X_tab DataFrame (12, 10) ['age', 'height_cm', 'score', 'hours', 'team_blue', 'team_green', 'team_red', 'device_laptop', 'device_phone', 'device_tablet']
Text LogisticRegression train accuracy: 1.0
Vocabulary size: 17
Image batch: (8, 3, 32, 32)
CNN logits: (8, 2)
One tiny training loss: 0.7139


## Stop Condition

A section counts as complete when:

- every check in that section passes;
- you can say the shape, dtype, and row meaning of the final object;
- any exported file reloads cleanly from disk;
- image augmentations preserve the correct label contract.
